In [1]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_NSIT_Dwarka_Delhi_CPCB_2023.xlsx")

In [5]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,257.0,230.0,212.0,105.0,176.0,185.0,66.0,105.0,136.0,191.0,362.0,377.0
1,2,385.0,272.0,299.0,189.0,123.0,275.0,103.0,NaN,137.0,194.0,392.0,354.0
2,3,391.0,279.0,NaN,289.0,116.0,136.0,138.0,100.0,130.0,188.0,473.0,309.0
3,4,351.0,NaN,188.0,142.0,76.0,266.0,212.0,112.0,131.0,219.0,387.0,317.0
4,5,294.0,NaN,214.0,222.0,233.0,334.0,94.0,120.0,111.0,237.0,439.0,327.0
5,6,390.0,228.0,252.0,205.0,324.0,263.0,91.0,132.0,96.0,272.0,NaN,300.0
6,7,342.0,283.0,282.0,259.0,164.0,328.0,66.0,135.0,75.0,223.0,NaN,297.0
7,8,344.0,324.0,290.0,303.0,170.0,316.0,74.0,150.0,87.0,152.0,412.0,314.0
8,9,452.0,293.0,165.0,306.0,254.0,277.0,59.0,139.0,97.0,187.0,421.0,355.0
9,10,385.0,224.0,318.0,275.0,294.0,226.0,50.0,178.0,73.0,182.0,266.0,333.0


In [6]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      36 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       33 non-null     float64
 7   July       36 non-null     float64
 8   August     34 non-null     float64
 9   September  28 non-null     float64
 10  October    34 non-null     float64
 11  November   32 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [7]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [8]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [9]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,257.0,230.00000,212.000000,105.0,176.0,185.0,66.000000,105.000000,136.0,191.0,362.0,377.0
1,2,385.0,272.00000,299.000000,189.0,123.0,275.0,103.000000,127.676471,137.0,194.0,392.0,354.0
2,3,391.0,279.00000,179.972222,289.0,116.0,136.0,138.000000,100.000000,130.0,188.0,473.0,309.0
3,4,351.0,252.59375,188.000000,142.0,76.0,266.0,89.305556,112.000000,131.0,219.0,387.0,317.0
4,5,294.0,252.59375,214.000000,222.0,233.0,334.0,94.000000,120.000000,111.0,237.0,439.0,327.0
